# 34_realtime_input_validation_JSC

- 목적: 22번에서 선정한 Ridge-VAR가 미래 샘플을 쓰지 않는 입력에서도 유지되는지 검증한다.
- 입력: 공통 형식 통일 후 `offline_mean`·`fixed_baseline`·`first_difference` 3종
- 계약: 10 Hz, 1초 윈도우, 과거 5샘플, burst 경계 상태 초기화, 정상 학습 q99 임계
- 출력: `figures/34_realtime_input_validation_JSC/`, `results/34_realtime_input_validation_JSC/`
- 담당: JSC

`offline_mean`은 22번 재현용 오프라인 상한이다. 배포 후보는 학습 정상 고정 상수와 1차 차분뿐이며, 고정 상수 결과는 이상 날짜 DC 교란이 남는 상한으로 해석한다.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
import paths
import realtime_jsc as R
NB = "34_realtime_input_validation_JSC"
FIG, RES = paths.nb_dirs(NB)
SEED = 0
np.random.seed(SEED)

## 1. 비교 계약

| 입력 | 시점 t에서 사용하는 정보 | 실시간성 | 주의 |
|---|---|---|---|
| `offline_mean` | 완성된 burst 전체 평균 | 불가 | 22번 기준선 재현 전용 |
| `fixed_baseline` | 학습 정상 긴 burst에서 고정한 상수 | 가능 | 이상 날짜의 잔여 DC가 라벨 교란일 수 있음 |
| `first_difference` | 현재 샘플 − 직전 샘플 | 가능 | burst 첫 샘플은 0으로 초기화 |

세 입력 모두 Ridge의 과거 버퍼를 burst 시작에서 비우고 첫 5샘플에는 점수를 내지 않는다. 첫 1초 윈도우 경보는 10번째 샘플이 들어오는 0.9초에 처음 가능하다. 공백은 보간하거나 이어 붙이지 않는다.

In [2]:
raw, unified, wins = R.load_inputs()
input_summary = pd.DataFrame({
    "item": ["samples", "segments", "1s windows", "normal windows", "outlier windows"],
    "value": [len(raw), raw['seg_uid'].nunique(), len(wins), (wins.label == 0).sum(), (wins.label == 1).sum()],
})
input_summary

,item,value
0,samples,20600
1,segments,620
2,1s windows,3360
3,normal windows,3264
4,outlier windows,96


## 2. 세 입력 × 9분할 실행

GroupKFold 5개와 정상 forward time-block 4개를 그대로 사용한다. baseline은 fold마다 학습 정상 UID로만 다시 추정한다. 처리시간은 형식 통일이 끝난 센서값을 Python에서 한 샘플씩 받아 baseline/차분 변환과 Ridge 예측까지 수행한 중앙값이다. 파일 I/O는 제외한다.

In [3]:
cv, scores, timing, baselines, diagnostics, errors, checks = R.run(
    NB, seed=SEED, lookback=5, repeats=2
)
print('cv rows:', len(cv), 'score rows:', len(scores), 'error rows:', len(errors))

cv rows: 27 score rows: 18810 error rows: 57


## 3. 계약 검증과 결과 저장

22번 `offline_mean` 수치 재현, burst별 cold-start, batch/샘플별 점수 정합성을 assert한다.

In [4]:
assert len(cv) == 3 * 9
assert checks['boundary_reset_ok'].all()
assert (checks['first_finite_index_min'] == 5).all()
assert (checks['first_finite_index_max'] == 5).all()
assert checks['batch_stream_max_relative_error'].max() < 1e-10

ref = pd.read_csv(ROOT / 'results/22_model_classical_timeseries_JSC/cv_scores.csv')
ref = ref[ref['key'] == 'ridge_var'].sort_values(['split', 'fold']).reset_index(drop=True)
now = cv[cv['variant'] == 'offline_mean'].sort_values(['split', 'fold']).reset_index(drop=True)
for col in ['auc', 'fpr_sample', 'fpr_segment', 'delay_median_s', 'n_detected']:
    assert np.allclose(now[col], ref[col], equal_nan=True), col

R.to_metrics(cv).to_csv(RES / 'metrics.csv', index=False)
cv.to_csv(RES / 'cv_scores.csv', index=False)
R.summary(cv).to_csv(RES / 'summary.csv', index=False)
scores.to_csv(RES / 'scores.csv', index=False)
timing.to_csv(RES / 'stream_timing.csv', index=False)
baselines.to_csv(RES / 'baseline_by_fold.csv', index=False)
diagnostics.to_csv(RES / 'input_diagnostics.csv', index=False)
errors.to_csv(RES / 'error_cases.csv', index=False)
checks.to_csv(RES / 'contract_checks.csv', index=False)
print('계약 검증 통과 · batch/stream 최대 절대오차', checks['batch_stream_max_abs_error'].max())
print('batch/stream 최대 상대오차', checks['batch_stream_max_relative_error'].max())

계약 검증 통과 · batch/stream 최대 절대오차 3.410605131648481e-13
batch/stream 최대 상대오차 6.772360450213455e-15


In [5]:
summary = R.summary(cv)
summary.round(6)

,variant,split,auc,fpr_sample,fpr_segment,n_detected,n_out_seg,delay_median_s,fpr_sample_high_load,fpr_sample_low_load,fpr_segment_high_load,fpr_segment_low_load,fit_sec,batch_score_sec
0,offline_mean,group_kfold_seg,1.0,0.011558,0.027999,3.4,3.4,0.9,0.023521,0.001648,0.056768,0.006154,0.201771,0.527037
1,fixed_baseline,group_kfold_seg,1.0,0.011254,0.024259,3.4,3.4,0.9,0.022921,0.001662,0.048687,0.006467,0.173901,0.548126
2,first_difference,group_kfold_seg,1.0,0.010036,0.020571,3.4,3.4,0.9,0.020887,0.001099,0.043232,0.003077,0.140615,0.396263
3,offline_mean,time_block,1.0,0.005971,0.014508,17.0,17.0,0.9,0.018418,0.000000,0.040036,0.000000,0.030592,0.076354
4,fixed_baseline,time_block,1.0,0.005540,0.012057,17.0,17.0,0.9,0.017520,0.000000,0.035134,0.000000,0.030397,0.084024
5,first_difference,time_block,1.0,0.006793,0.016959,17.0,17.0,0.9,0.020057,0.000000,0.044938,0.000000,0.029608,0.079669


## 4. 잔여 DC 교란과 처리시간

잔여 세그먼트 평균의 AUC는 성능을 높이기 위한 피처가 아니라 입력에 날짜/파일 교란이 남았는지 확인하는 진단이다. `fixed_baseline`의 실제 이상 성능은 이 수치가 높으면 상한으로만 읽는다.

In [6]:
dc_summary = diagnostics.groupby(['variant', 'channel'], sort=False)['abs_segment_mean_auc'].agg(['mean', 'min', 'max']).reset_index()
time_summary = timing.groupby(['variant', 'split'], sort=False).agg(
    us_per_sample=('us_per_received_sample', 'median'),
    throughput_hz=('throughput_received_hz', 'median'),
).reset_index()
display(dc_summary.round(6))
display(time_summary.round(3))

,variant,channel,mean,min,max
0,offline_mean,AI0_Vibration,0.500000,0.500000,0.500000
1,offline_mean,AI1_Vibration,0.500000,0.500000,0.500000
2,offline_mean,AI2_Current,0.500000,0.500000,0.500000
3,fixed_baseline,AI0_Vibration,0.959156,0.959156,0.959156
4,fixed_baseline,AI1_Vibration,0.947947,0.947947,0.947947
5,fixed_baseline,AI2_Current,0.919706,0.919645,0.919978
6,first_difference,AI0_Vibration,0.722420,0.722420,0.722420
7,first_difference,AI1_Vibration,0.792564,0.792564,0.792564
8,first_difference,AI2_Current,0.563263,0.563263,0.563263


,variant,split,us_per_sample,throughput_hz
0,offline_mean,group_kfold_seg,342.528,2919.469
1,fixed_baseline,group_kfold_seg,339.297,2947.273
2,first_difference,group_kfold_seg,299.561,3338.214
3,offline_mean,time_block,73.237,13657.959
4,fixed_baseline,time_block,74.363,13451.035
5,first_difference,time_block,72.651,13764.832


In [7]:
R.plot_comparison(cv, FIG)
R.plot_state_fpr(cv, FIG)
R.plot_timing(timing, FIG)
print('saved', FIG.relative_to(ROOT))

saved figures\34_realtime_input_validation_JSC


## 5. causal 입력 판정

합격 기준은 time-block에서 (1) 17/17 유지, (2) AUC ≥ 0.99, (3) 오프라인 mean 대비 세그먼트 오경보율 증가가 0.01p 이내다. baseline은 기준을 통과해도 잔여 DC 교란 때문에 최종 주 입력으로 자동 채택하지 않는다.

In [8]:
tb = summary[summary['split'] == 'time_block'].set_index('variant')
offline_fpr = float(tb.loc['offline_mean', 'fpr_segment'])
verdict = pd.DataFrame([
    {"variant": v,
     "causal": v != 'offline_mean',
     "detected_17_of_17": bool(np.isclose(tb.loc[v, 'n_detected'], 17)),
     "auc_ge_099": bool(tb.loc[v, 'auc'] >= 0.99),
     "fpr_within_offline_plus_001": bool(tb.loc[v, 'fpr_segment'] <= offline_fpr + 0.01),
     "decision": ("offline benchmark" if v == 'offline_mean' else
                  "upper bound: residual DC" if v == 'fixed_baseline' else
                  "adopt" if (np.isclose(tb.loc[v, 'n_detected'], 17) and tb.loc[v, 'auc'] >= 0.99 and tb.loc[v, 'fpr_segment'] <= offline_fpr + 0.01) else "reject")}
    for v in R.VARIANTS
])
verdict.to_csv(RES / 'verdict.csv', index=False)
verdict

,variant,causal,detected_17_of_17,auc_ge_099,fpr_within_offline_plus_001,decision
0,offline_mean,False,True,True,True,offline benchmark
1,fixed_baseline,True,True,True,True,upper bound: residual DC
2,first_difference,True,True,True,True,adopt


## 6. 해석 한계

- 실제 이상은 한 날짜·한 이벤트이므로 17개 burst는 독립 고장 17건이 아니다.
- `fixed_baseline`은 causal이지만 이상 파일의 DC 오프셋을 남긴다. 이 입력의 실제 이상 AUC는 파형 성능으로 단정하지 않는다.
- 처리시간은 현재 PC·Python·단일 샘플 `sklearn.predict` 기준이며 센서 통신·파일 I/O는 포함하지 않는다.
- 1차 차분 첫 값 0은 명시적인 burst 시작값이다. 경계마다 초기화하며 다른 burst의 마지막 값과 차분하지 않는다.
- 첫 시점 점수는 0.5초에 계산 가능하지만 공통 1초 윈도우 판정은 0.9초에 나온다.